# SMART — Random-Walk Graph VisualisationVisualises label propagation for **one window of one unit** in three panels thatmirror the method figure:1. **Seeds** — the ~10% of frames that keep their label, coloured by class;   everything else is grey and unknown.2. **Propagated** — every frame coloured by the class whose random walk reached   it with the most probability mass.3. **Ground truth** — the same window coloured by its true labels, for comparison.Node positions come from one spring layout computed once and reused by all threepanels, so the eye can track a node across them. Edges are the sparsifiedaffinity graph the method actually walks on.Requires `networkx` and `plotly`.

In [ ]:
import sys, picklefrom pathlib import Pathimport numpy as npimport networkx as nximport plotly.graph_objects as goimport colorsyssys.path.insert(0, '..')from tools.graph import (normalize_features, pdf_weights, sparsify,                         add_temporal_edges, transition_matrix, propagate)# ---- edit these -------------------------------------------------------DATA_DIR    = Path('../artifacts/adl')UNIT        = 'P_11'FEAT_SUFFIX = '_resnet'N           = 100      # frames in the windowWIN_START   = 0        # first frame of the window (0-based internal index)# ----------------------------------------------------------------------FEAT_NORM='center'; SIGMA_MODE='median'; GAMMA=0.90; RW_STEPS=10   # match evaluate.pyLAYOUT_SEED = 10       # fixes node positions so all three panels alignprint('viz config:', dict(UNIT=UNIT, N=N, WIN_START=WIN_START,                          GAMMA=GAMMA, RW_STEPS=RW_STEPS))

## Load one window and build the graph the method actually walks on

In [ ]:
names   = pickle.load(open(DATA_DIR/'class_map.pkl','rb'))['names']d       = np.load(DATA_DIR/f'{UNIT}_data.npz', allow_pickle=True)feats   = np.load(DATA_DIR/f'{UNIT}_feats{FEAT_SUFFIX}.npz')['feats']Yall, seedsall = d['targets'], d['seeds']end = min(WIN_START + N, feats.shape[0])idx = np.arange(WIN_START, end)F   = normalize_features(feats[idx], FEAT_NORM)Yw, seedw = Yall[idx], seedsall[idx]S, D, W, sigma = pdf_weights(F, sigma_mode=SIGMA_MODE)A = add_temporal_edges(sparsify(W, 'cdf', GAMMA))P = transition_matrix(A)scores = propagate(P, Yw, seedw, RW_STEPS)n = len(idx)print(f'window: {n} frames | seeds={int(seedw.sum())} '      f'| classes present={int((Yw.sum(0)>0).sum())} | sigma={sigma:.3f}')def primary(row):    nz = np.where(row > 0)[0]    return int(nz[0]) if len(nz) else -1      # -1 = background / unlabelledgt_all   = {i: primary(Yw[i]) for i in range(n)}gt_seed  = {i: primary(Yw[i]) for i in range(n) if seedw[i]}pred_cls = scores.argmax(1)active   = scores.max(1) > 0

In [ ]:
G = nx.Graph()G.add_nodes_from(range(n))iu = np.triu_indices(n, k=1)for a, b in zip(*iu):    if A[a, b] > 0:        G.add_edge(int(a), int(b), weight=float(A[a, b]))pos = nx.spring_layout(G, weight='weight', seed=LAYOUT_SEED, k=1.5/np.sqrt(n))print(f'graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges '      f'(after CDF sparsification + temporal chain)')

## Drawing helpersColours are spaced by the golden-ratio hue increment so consecutive class idsstay visually distinct — a plain palette with modulo wrap collides as soon as theclass count exceeds the palette length.

In [ ]:
_present = sorted({c for c in gt_all.values() if c >= 0}                  | {int(pred_cls[i]) for i in range(n) if active[i]})def _distinct_colors(class_ids):    cmap = {}    for j, cls in enumerate(class_ids):        h = (j * 0.618033988749895) % 1.0        r, g, b = colorsys.hsv_to_rgb(h, 0.78, 0.88)        cmap[int(cls)] = f'rgb({int(r*255)},{int(g*255)},{int(b*255)})'    return cmapCLASS_COLOR = _distinct_colors(_present)GREY = '#c9d6df'def color_of(cls):    cls = int(cls)    return GREY if cls < 0 else CLASS_COLOR.get(cls, '#808080')def edge_trace():    xs, ys = [], []    for a, b in G.edges():        xs += [pos[a][0], pos[b][0], None]        ys += [pos[a][1], pos[b][1], None]    return go.Scatter(x=xs, y=ys, mode='lines',                      line=dict(width=0.3, color='rgba(110,160,190,0.25)'),                      hoverinfo='none', showlegend=False)def node_trace(colors, sizes, hover):    return go.Scatter(x=[pos[i][0] for i in range(n)],                      y=[pos[i][1] for i in range(n)],                      mode='markers',                      marker=dict(color=colors, size=sizes,                                  line=dict(width=0.5, color='white')),                      hoverinfo='text', hovertext=hover, showlegend=False)def make_fig(title, colors, sizes, hover, legend_ids=None):    fig = go.Figure(data=[edge_trace(), node_trace(colors, sizes, hover)])    if legend_ids:        for cls in legend_ids:            fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',                                     marker=dict(size=9, color=color_of(cls)),                                     name=str(names[cls])[:28], showlegend=True))    fig.update_layout(title=title, template='simple_white',                      xaxis=dict(visible=False), yaxis=dict(visible=False),                      width=780, height=620, margin=dict(l=10, r=10, t=48, b=10),                      legend=dict(font=dict(size=10)))    return fig

### Panel 1 — seeds onlyThe input to propagation: a handful of coloured nodes in a sea of grey.

In [ ]:
colors = [color_of(gt_seed.get(i, -1)) for i in range(n)]sizes  = [11 if i in gt_seed else 5 for i in range(n)]hover  = [f'frame {int(idx[i])}<br>' +          (f'SEED: {names[gt_seed[i]]}' if i in gt_seed else 'unlabelled')          for i in range(n)]make_fig(f'{UNIT} — panel 1: {int(seedw.sum())} seed frames',         colors, sizes, hover, legend_ids=sorted(set(gt_seed.values()) - {-1})).show()

### Panel 2 — after propagationEvery frame now carries a class. Node size encodes how much probability mass thewinning walk delivered — small nodes are the uncertain ones.

In [ ]:
mx = scores.max(1)norm = (mx - mx.min()) / (np.ptp(mx) + 1e-12)colors = [color_of(pred_cls[i]) if active[i] else GREY for i in range(n)]sizes  = [5 + 9*float(norm[i]) for i in range(n)]hover  = [f'frame {int(idx[i])}<br>pred: {names[int(pred_cls[i])]}'          f'<br>score: {scores[i].max():.3f}'          f'<br>true: {", ".join(names[c] for c in np.where(Yw[i]>0)[0]) or "-"}'          for i in range(n)]make_fig(f'{UNIT} — panel 2: after {RW_STEPS}-step random walk',         colors, sizes, hover,         legend_ids=sorted({int(pred_cls[i]) for i in range(n) if active[i]})).show()

### Panel 3 — ground truthCompare against panel 2. Nodes with a black ring carry two or more concurrentactions — the frames that a single-label method cannot represent at all.

In [ ]:
colors = [color_of(gt_all[i]) for i in range(n)]sizes  = [9 if Yw[i].sum() > 0 else 5 for i in range(n)]hover  = [f'frame {int(idx[i])}<br>true: '          f'{", ".join(names[c] for c in np.where(Yw[i]>0)[0]) or "background"}'          for i in range(n)]fig = make_fig(f'{UNIT} — panel 3: ground truth',               colors, sizes, hover,               legend_ids=sorted(set(gt_all.values()) - {-1}))multi = [i for i in range(n) if Yw[i].sum() >= 2]if multi:    fig.add_trace(go.Scatter(x=[pos[i][0] for i in multi],                             y=[pos[i][1] for i in multi],                             mode='markers',                             marker=dict(size=15, color='rgba(0,0,0,0)',                                         line=dict(width=1.6, color='black')),                             name=f'>=2 concurrent actions ({len(multi)})',                             hoverinfo='skip'))fig.show()

### Optional — dense W vs sparsified AQuantifies what CDF pruning removes. The dense affinity graph is nearly complete;after pruning, each frame keeps only the neighbours holding `gamma` of itsprobability mass, and the walk becomes meaningful.

In [ ]:
dense_edges  = int((W > 0).sum() - n) // 2sparse_edges = G.number_of_edges()print(f'dense W        : {dense_edges:>7} edges')print(f'CDF-sparsified : {sparse_edges:>7} edges  '      f'({100*sparse_edges/max(dense_edges,1):.1f}% kept)')print(f'mean degree    : {2*sparse_edges/n:.1f} (dense would be {n-1})')deg = np.asarray((A > 0).sum(1)).ravel() - 1print(f'degree after pruning: min={deg.min()} median={np.median(deg):.0f} max={deg.max()}')

In [ ]:
# Export the three panels for the paper (kaleido required for static export):#   pip install kaleido# fig.write_image('../images/graph_panel3.pdf', width=780, height=620, scale=2)